![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 2 -- Lab 1: Toolkit Walkthrough

The U.S. Census Bureau collects demographic data on every American. A government agency wants to understand which factors most strongly predict whether an individual earns above or below $50K per year.

**Your role:** Work through this notebook cell by cell. By the end you will be comfortable with Pandas, Matplotlib, Seaborn, and Scikit-learn's preprocessing tools -- everything you need to prepare real data for a machine learning model.

**Dataset:** UCI Adult Census Income (48,842 records, 14 features)

---
## 1. Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')

---
## 2. Loading the Data

We use `kagglehub` to download the UCI Adult Census Income dataset directly. The function returns a local directory path where the files are stored.

In [ ]:
import kagglehub

path = kagglehub.dataset_download('uciml/adult-census-income')
print('Dataset downloaded to:', path)
print('Files:', os.listdir(path))

In [ ]:
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
df.head(10)

In [ ]:
print('Shape:', df.shape)
print()
print('Columns:', df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

### Understanding dtypes

Pandas assigns a **dtype** (data type) to every column:

| dtype | Meaning | Examples in this dataset |
|---|---|---|
| `int64` | Whole numbers | age, education.num, capital.gain |
| `float64` | Decimal numbers | (none here, but common after imputation) |
| `object` | Text / strings | workclass, education, income |

Numeric columns can be used directly in math and plotting. Object columns need encoding before a model can use them.

In [ ]:
df.dtypes

### Stripping whitespace from string columns

This dataset has leading spaces in many categorical values (e.g. `' Private'` instead of `'Private'`). We strip them so that filtering and grouping work correctly.

In [ ]:
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)
df.head()

---
## 3. Pandas Essentials

### `.loc[]` -- Label-based selection

`.loc[row_condition, column_names]` selects rows by a boolean condition and columns by their string names. It is the primary way to filter a DataFrame.

In [ ]:
senior_high_earners = df.loc[(df['age'] > 50) & (df['income'] == '>50K')]
print('Rows matching (age > 50 AND income >50K):', len(senior_high_earners))
senior_high_earners.head()

### `.iloc[]` -- Integer-position selection

`.iloc[row_positions, column_positions]` selects by integer index. Useful when you want a specific slice by position rather than by label.

In [ ]:
df.iloc[:5, -3:]

### Boolean filtering with `&` and `|`

Combine conditions with `&` (AND) and `|` (OR). Each condition must be in parentheses because of Python operator precedence.

In [ ]:
young_or_old = df[(df['age'] < 25) | (df['age'] > 60)]
print('People younger than 25 or older than 60:', len(young_or_old))
young_or_old.head()

### `.value_counts()` -- Frequency of each unique value

Returns a Series sorted by count descending. Pass `normalize=True` to get proportions instead of raw counts.

In [ ]:
print('--- workclass ---')
print(df['workclass'].value_counts())
print()
print('--- education ---')
print(df['education'].value_counts())
print()
print('--- income ---')
print(df['income'].value_counts())

### `.groupby()` -- Split-Apply-Combine

`.groupby(column)` splits the DataFrame into groups. You then apply an aggregation (mean, sum, count, etc.) to each group. The result is a new Series or DataFrame.

In [ ]:
df.groupby('education')['hours.per.week'].mean().sort_values(ascending=False)

### `.sort_values()` -- Sort rows by a column

Pass `ascending=False` to sort largest first. Chain `.head(n)` to keep only the top n rows.

In [ ]:
df.sort_values('age', ascending=False).head(10)

### `.apply()` -- Apply a custom function to each element

`.apply()` runs a function on every value in a Series (column). Lambda functions are convenient for short transformations.

In [ ]:
df['age_group'] = df['age'].apply(lambda x: 'Senior' if x > 50 else 'Junior')
df[['age', 'age_group']].head(10)

### `.nunique()` -- Count distinct values per column

Useful for a quick overview of cardinality. Columns with very many unique values (like `fnlwgt`) may need special handling.

In [ ]:
df.nunique()

---
## 4. Matplotlib Fundamentals

Matplotlib is the foundational plotting library in Python. Seaborn and Pandas plotting are built on top of it.

### Histogram

- `figsize=(width, height)` controls the figure size in inches.
- `bins` controls how many bars the data is split into. More bins = finer detail.
- `edgecolor` draws a border around each bar so they are visually distinct.
- `alpha` sets transparency (0 = invisible, 1 = opaque).

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['age'], bins=40, edgecolor='black', alpha=0.7)
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Distribution of Age')
plt.show()

### Scatter plot

- `alpha=0.1` makes each point nearly transparent, so dense regions appear darker. This is essential when you have tens of thousands of points (overplotting).
- `s=10` controls the marker size in points squared.

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(df['age'], df['hours.per.week'], alpha=0.1, s=10)
plt.xlabel('Age')
plt.ylabel('Hours per Week')
plt.title('Age vs Hours per Week')
plt.show()

### Bar chart

Bar charts are best for comparing counts or means across categories. Here we show the top 10 education levels by frequency.

In [ ]:
edu_counts = df['education'].value_counts().head(10)

plt.figure(figsize=(10, 5))
plt.bar(edu_counts.index, edu_counts.values, edgecolor='black', alpha=0.7)
plt.xlabel('Education')
plt.ylabel('Count')
plt.title('Top 10 Education Levels')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Subplots

`plt.subplots(nrows, ncols)` creates a grid of axes. Each axis is an independent plot. This lets you compare multiple distributions side by side.

`plt.tight_layout()` adjusts spacing so labels do not overlap.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df['age'], bins=40, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Age')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Count')

axes[0, 1].hist(df['hours.per.week'], bins=40, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Hours per Week')
axes[0, 1].set_xlabel('Hours per Week')
axes[0, 1].set_ylabel('Count')

axes[1, 0].hist(df['education.num'], bins=16, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('Education Num')
axes[1, 0].set_xlabel('Education Number')
axes[1, 0].set_ylabel('Count')

axes[1, 1].hist(df['capital.gain'], bins=50, edgecolor='black', alpha=0.7, color='red')
axes[1, 1].set_title('Capital Gain')
axes[1, 1].set_xlabel('Capital Gain')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## 5. Seaborn Fundamentals

Seaborn wraps Matplotlib with a higher-level API designed for statistical visualization. It integrates directly with Pandas DataFrames.

### `histplot` with KDE overlay

- `hue` colors the bars by a categorical column, automatically creating a legend.
- `kde=True` overlays a Kernel Density Estimate -- a smooth curve that approximates the probability density function. This makes it easier to see the overall shape.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='income', kde=True, bins=40)
plt.title('Age Distribution by Income')
plt.show()

### Boxplot

A boxplot summarizes a distribution with five numbers:
- **Line inside the box** = median (50th percentile)
- **Box edges** = Q1 (25th percentile) and Q3 (75th percentile). The box height is the IQR (Interquartile Range = Q3 - Q1).
- **Whiskers** extend up to 1.5 * IQR from the box edges.
- **Dots beyond the whiskers** = outliers.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='income', y='age')
plt.title('Age by Income Level')
plt.show()

### Violin plot

A violin plot is like a boxplot, but it also shows the **density** of the data at each value. Wider sections mean more data points. It reveals multimodality (multiple peaks) that a boxplot hides.

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x='income', y='hours.per.week')
plt.title('Hours per Week by Income Level')
plt.show()

### Count plot

`countplot` is a bar chart that counts observations in each category. The `order` parameter lets you sort bars by frequency. `hue` splits each bar by a second categorical variable.

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df, x='workclass', hue='income',
              order=df['workclass'].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.title('Workclass Counts by Income')
plt.tight_layout()
plt.show()

### Correlation heatmap

- `annot=True` prints the correlation coefficient in each cell.
- `cmap='coolwarm'` uses blue for negative and red for positive correlations.
- `fmt='.2f'` formats numbers to 2 decimal places.
- `linewidths=0.5` adds thin borders between cells.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

### Pair plot

A pairplot draws scatterplots for every pair of numeric columns and distributions on the diagonal. `hue` colors points by class.

We sample 1,000 rows because pairplots are slow on the full dataset (48K rows x multiple panels). `random_state=42` makes the sample reproducible.

In [ ]:
subset = df[['age', 'hours.per.week', 'education.num', 'capital.gain', 'income']].sample(1000, random_state=42)
sns.pairplot(subset, hue='income', diag_kind='kde')
plt.suptitle('Pair Plot (sample of 1000)', y=1.02)
plt.show()

### Scatter plot with multiple encodings

`hue` maps color to income, and `style` maps marker shape to sex. This lets you see two categorical dimensions on a single scatter plot. Again we sample for speed.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(2000, random_state=42),
                x='age', y='hours.per.week',
                hue='income', style='sex', alpha=0.6)
plt.title('Age vs Hours per Week (colored by Income, shaped by Sex)')
plt.show()

---
## 6. Pandas Built-in Plotting

Pandas Series and DataFrames have a `.plot` accessor that calls Matplotlib under the hood. It is the fastest way to get a quick visual during exploratory analysis.

In [ ]:
df['age'].plot.hist(bins=30, figsize=(8, 4), title='Age Distribution', edgecolor='black')
plt.xlabel('Age')
plt.show()

In [ ]:
df['workclass'].value_counts().plot.bar(figsize=(10, 4), rot=45, color='steelblue')
plt.title('Workclass Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
df.plot.scatter(x='age', y='hours.per.week', alpha=0.1, figsize=(8, 5))
plt.title('Age vs Hours per Week')
plt.show()

### When to use which library?

| Task | Recommended library |
|---|---|
| Quick one-line exploration | **Pandas** `.plot` |
| Publication-quality statistical plots | **Seaborn** |
| Full customization, annotations, subplots | **Matplotlib** |

In practice you combine all three: Seaborn for the core plot, Matplotlib for fine-tuning, and Pandas for fast checks.

---
## 7. How to Read Plots

### Histograms

A histogram shows how values are **distributed** across bins.

- **Symmetric** (bell-shaped): values are evenly spread around the center (e.g., heights of adults).
- **Right-skewed**: a long tail extends to the right. Most values are low, but a few are very high (e.g., income, capital gains).
- **Bimodal**: two distinct peaks, suggesting two subpopulations mixed together.
- **The peak** (mode) tells you the most common range of values.

### Boxplots

- **Median line** (inside the box): the middle value. Half the data is above, half below.
- **Box** (Q1 to Q3): contains the middle 50% of the data. The height of the box is the **IQR** (Interquartile Range).
- **Whiskers**: extend up to **1.5 * IQR** from Q1 and Q3. They show the range of "typical" values.
- **Dots beyond whiskers**: individual outliers -- unusually high or low values.

### Correlation heatmaps

Each cell shows the Pearson correlation coefficient between two numeric features.

- **+1** = perfect positive linear relationship (as X goes up, Y goes up proportionally).
- **-1** = perfect negative linear relationship (as X goes up, Y goes down proportionally).
- **0** = no linear relationship (X and Y vary independently).

Values close to +/-1 indicate strong relationships. Values near 0 indicate weak or no linear relationship (but there could still be a nonlinear one).

### Pairplots

- **Diagonal panels** show the distribution of each individual feature (histogram or KDE).
- **Off-diagonal panels** show the pairwise scatter plot between two features.
- **Color separation**: if the colored classes form distinct clusters in a panel, that pair of features helps the model distinguish the classes. If the colors overlap completely, those features are not useful for separation.

### Annotated boxplot example

The following cell creates a boxplot with text annotations pointing to each component, so you can see exactly where the median, quartiles, whiskers, and outliers are.

In [ ]:
data = df['age']
q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
median = data.median()
iqr = q3 - q1
whisker_low = max(data.min(), q1 - 1.5 * iqr)
whisker_high = min(data.max(), q3 + 1.5 * iqr)

fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(data, vert=True, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor='lightblue', edgecolor='black'),
                medianprops=dict(color='red', linewidth=2))

ax.annotate(f'Median = {median:.0f}',
            xy=(1, median), xytext=(1.3, median),
            fontsize=11, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

ax.annotate(f'Q1 = {q1:.0f}',
            xy=(1, q1), xytext=(1.3, q1 - 3),
            fontsize=11, color='blue',
            arrowprops=dict(arrowstyle='->', color='blue'))

ax.annotate(f'Q3 = {q3:.0f}',
            xy=(1, q3), xytext=(1.3, q3 + 3),
            fontsize=11, color='blue',
            arrowprops=dict(arrowstyle='->', color='blue'))

ax.annotate(f'Whisker low = {whisker_low:.0f}',
            xy=(1, whisker_low), xytext=(1.3, whisker_low - 3),
            fontsize=11, color='green',
            arrowprops=dict(arrowstyle='->', color='green'))

ax.annotate(f'Whisker high = {whisker_high:.0f}',
            xy=(1, whisker_high), xytext=(1.3, whisker_high + 3),
            fontsize=11, color='green',
            arrowprops=dict(arrowstyle='->', color='green'))

outliers = data[(data < whisker_low) | (data > whisker_high)]
if len(outliers) > 0:
    ax.annotate(f'Outliers ({len(outliers)} points)',
                xy=(1, outliers.max()), xytext=(1.3, outliers.max() + 3),
                fontsize=11, color='purple',
                arrowprops=dict(arrowstyle='->', color='purple'))

ax.set_title('Annotated Boxplot of Age', fontsize=14)
ax.set_ylabel('Age')
ax.set_xticks([])
plt.tight_layout()
plt.show()

---
## 8. Handling Missing Values

Real-world data is messy. In this dataset, missing values are encoded as the string `'?'` rather than as proper NaN. We first convert them, then decide how to fill them.

In [ ]:
df.replace('?', np.nan, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
print('Percentage of missing values per column:')
print(missing_pct)

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, cmap='magma', yticklabels=False)
plt.title('Missing Value Heatmap (yellow = missing)')
plt.tight_layout()
plt.show()

### Imputation with mode

For categorical columns with missing values (workclass, occupation, native.country), we fill each missing value with the **mode** -- the most frequent value in that column. This is a simple and reasonable default.

In [ ]:
for col in ['workclass', 'occupation', 'native.country']:
    df[col].fillna(df[col].mode()[0], inplace=True)

print('Missing values after imputation:')
print(df.isnull().sum())

---
## 9. Feature Scaling

Models like SVM and KNN use distance. If `capital.gain` ranges 0--99999 but `age` ranges 17--90, the model is biased toward capital.gain because its raw numbers are much larger. Scaling puts all features on a comparable range.

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
print('Numeric columns:', num_cols)

In [ ]:
df[num_cols].describe().loc[['min', 'max']]

### MinMaxScaler

Squashes every feature to the range [0, 1]. The formula is: `(x - min) / (max - min)`. Good when you need bounded values.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(minmax_scaler.fit_transform(df[num_cols]),
                         columns=num_cols)
df_minmax.describe()

### StandardScaler

Centers each feature at mean=0 with std=1. The formula is: `(x - mean) / std`. Good when features are roughly Gaussian.

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
df_standard = pd.DataFrame(std_scaler.fit_transform(df[num_cols]),
                           columns=num_cols)
df_standard.describe()

### Side-by-side comparison: capital.gain before and after scaling

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(df['capital.gain'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Original capital.gain')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count')

axes[1].hist(df_minmax['capital.gain'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('MinMax Scaled capital.gain')
axes[1].set_xlabel('Value [0, 1]')
axes[1].set_ylabel('Count')

axes[2].hist(df_standard['capital.gain'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[2].set_title('Standard Scaled capital.gain')
axes[2].set_xlabel('Value (mean=0, std=1)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

### When to use which scaler?

| Scaler | Range | Best for |
|---|---|---|
| **MinMaxScaler** | [0, 1] | When you need bounded values (e.g., neural networks with sigmoid output) |
| **StandardScaler** | mean=0, std=1 | When features are roughly Gaussian (e.g., SVM, logistic regression) |

**Tree-based models** (Decision Trees, Random Forests, Gradient Boosting) do **not** need scaling because they split on thresholds, not distances.

---
## 10. Encoding Categorical Features

ML models do math. They cannot process `'Private'` or `'Male'`. We convert categories to numbers.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Object columns:', cat_cols)

### LabelEncoder (for binary target)

`LabelEncoder` maps each unique string to an integer. It is suitable for the **target** column or ordinal features where order matters.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['income_encoded'] = le.fit_transform(df['income'])

print('Label mapping:')
for label, encoded in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {label} -> {encoded}')

df[['income', 'income_encoded']].head(10)

### One-Hot Encoding with `pd.get_dummies()`

For **nominal** features (no natural order, like workclass or marital status), we create a binary column for each category. `drop_first=True` drops one column per feature to avoid the **dummy variable trap** (perfect multicollinearity).

In [ ]:
df_dummies = pd.get_dummies(df, columns=['workclass', 'marital.status'], drop_first=True)
print('Shape after one-hot encoding:', df_dummies.shape)
print()
new_cols = [c for c in df_dummies.columns if c.startswith('workclass_') or c.startswith('marital.status_')]
print('New columns created:')
print(new_cols)

### Label encoding vs One-hot encoding

| Method | When to use | Example |
|---|---|---|
| **Label encoding** | Feature has a natural order (ordinal) | education level: HS < Bachelors < Masters |
| **One-hot encoding** | Feature has no natural order (nominal) | workclass: Private, Self-emp, Federal-gov |

`drop_first=True` avoids the dummy variable trap: if you know all other dummies are 0, the dropped one must be 1, so it carries no extra information.

---
## 11. Train-Test Split

We hold out 20% of the data as a **test set** that the model never sees during training. This lets us evaluate how well the model generalizes to new, unseen data.

In [ ]:
y = df['income_encoded']

cols_to_drop = ['income', 'income_encoded', 'age_group']
X = df.drop(columns=cols_to_drop)

X = pd.get_dummies(X, drop_first=True)

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape: ', y_test.shape)

In [ ]:
print('Training set class proportions:')
print(y_train.value_counts(normalize=True))
print()
print('Test set class proportions:')
print(y_test.value_counts(normalize=True))

### Key parameters

- `test_size=0.2` -- 20% of the data goes to the test set, 80% to training.
- `random_state=42` -- a seed for the random split, ensuring reproducibility. Anyone running this notebook gets the exact same split.
- `stratify=y` -- preserves the class proportions in both sets. If the original data is 75% low income / 25% high income, both train and test will have the same ratio.

---
## 12. The Scikit-learn API

Every scikit-learn estimator (scaler, encoder, model) follows the same three-method pattern:

1. **`fit(X_train)`** -- learn parameters from training data (e.g., mean and std for StandardScaler).
2. **`transform(X)`** -- apply the learned parameters to any data.
3. **`fit_transform(X_train)`** -- shortcut that does fit + transform in one call.

**Critical rule:** NEVER fit on test data. The scaler/encoder must learn its parameters only from the training set, then apply (transform) them to the test set.

In [ ]:
scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('X_train_scaled shape:', X_train_scaled.shape)
print('X_test_scaled shape: ', X_test_scaled.shape)
print()
print('X_train_scaled mean (first 5 features):', X_train_scaled[:, :5].mean(axis=0).round(4))
print('X_train_scaled std  (first 5 features):', X_train_scaled[:, :5].std(axis=0).round(4))

### Pipeline

A `Pipeline` chains multiple steps into a single object. This prevents data leakage and keeps your code clean. Each step is a `(name, estimator)` tuple.

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler())
])

X_train_piped = pipe.fit_transform(X_train)

print('Shape after pipeline:', X_train_piped.shape)
print('Mean (first 5 features):', X_train_piped[:, :5].mean(axis=0).round(4))
print('Std  (first 5 features):', X_train_piped[:, :5].std(axis=0).round(4))

Tomorrow we add a model as the second step in this pipeline.

---
## 13. Summary

You now have hands-on experience with the core data science toolkit:

- **Pandas** -- loading, inspecting, filtering, grouping, and transforming tabular data
- **Matplotlib** -- creating figures, histograms, scatter plots, bar charts, and subplots
- **Seaborn** -- statistical visualizations: distributions, boxplots, heatmaps, pairplots
- **Scikit-learn** -- scaling (MinMaxScaler, StandardScaler), encoding (LabelEncoder, OneHotEncoder), splitting (train_test_split), and the fit/transform API

The Adult Census dataset is now cleaned, encoded, scaled, and split into training and test sets. It is ready for a predictive model -- which we will build in the next sessions.